# Market Risk Analytics: SQL & PowerBI Project
# By: Samuel Reano

A market risk analytics project that focuses on rigorous multi-period risk profiling, blending historical simulated risk metrics with dynamic sector shock modeling. By leveraging flexible DAX measures (in PowerBI) and SQLite views, this project delivers granular visibility into tail-risk distributions, Expected Shortfall, Value-at-Risk, and individual security contributions under extreme market conditions. In this notebook, we create a market risk database using Python and SQL together.

## Import Python Libraries and SQLite

In [ ]:
import sqlite3
import pandas as pd
import yfinance as yf
import numpy as np

## Creating the market risk database in Python and SQLite


In [ ]:
# 1. Download Raw Market Data from Yahoo Finance
tickers = [
    "AAPL",
    "MSFT",
    "JPM",
    "XOM",
    "JNJ",
    "WMT",
    "CAT",
    "NEE",
    "NVDA",
    "KO",
]

frames = []
for ticker in tickers:
  df = yf.download(
      ticker,
      start="2024-04-09",
      end="2026-04-09",
      interval="1d",
      auto_adjust=False,
      multi_level_index=False,
      progress=False,
  )
  df = df.reset_index()
  df["Ticker"] = ticker
  frames.append(df)

market_data = pd.concat(frames, ignore_index=True)

# Clean and Standardize Columns
market_data.columns = [str(col).lower() for col in market_data.columns]
market_data = market_data.rename(
    columns={
        "date": "trade_date",
        "close": "close_price",
        "open": "open_price",
        "high": "high_price",
        "low": "low_price",
        "volume": "volume",
        "ticker": "ticker",
    }
)
market_data["trade_date"] = pd.to_datetime(
    market_data["trade_date"]
).dt.strftime("%Y-%m-%d")

# Define Metadata
sector_mapping = {
    "AAPL": "Technology",
    "MSFT": "Technology",
    "JPM": "Financials",
    "XOM": "Energy",
    "JNJ": "Healthcare",
    "WMT": "Consumer Staples",
    "CAT": "Industrials",
    "NEE": "Utilities",
    "NVDA": "Technology",
    "KO": "Consumer Staples",
}

shares_mapping = {
    "AAPL": 150,
    "MSFT": 100,
    "JPM": 200,
    "XOM": 250,
    "JNJ": 120,
    "WMT": 180,
    "CAT": 80,
    "NEE": 300,
    "NVDA": 130,
    "KO": 220,
}

portfolio_df = pd.DataFrame({
    "ticker": list(sector_mapping.keys()),
    "sector": list(sector_mapping.values()),
    "shares_held": [shares_mapping[t] for t in sector_mapping.keys()],
})

# 2. Establish SQLite Connection & Push Raw Data First
db_name = "market_risk.db"
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Enable foreign key support in SQLite
cursor.execute("PRAGMA foreign_keys = ON;")

# Drop existing tables if re-running script
cursor.execute("DROP VIEW IF EXISTS vw_daily_portfolio_exposure;")
cursor.execute("DROP TABLE IF EXISTS asset_volatility;")
cursor.execute("DROP TABLE IF EXISTS risk_metrics;")
cursor.execute("DROP TABLE IF EXISTS historical_prices;")
cursor.execute("DROP TABLE IF EXISTS portfolio_mapping;")

# Create Portfolio Mapping Table (Dimension Table)
cursor.execute("""
CREATE TABLE portfolio_mapping (
    ticker TEXT PRIMARY KEY,
    sector TEXT NOT NULL,
    shares_held REAL NOT NULL
);
""")

# Create Historical Prices Table (Fact Table) with constraints
cursor.execute("""
CREATE TABLE historical_prices (
    trade_date TEXT NOT NULL,
    ticker TEXT NOT NULL,
    open_price REAL,
    high_price REAL,
    low_price REAL,
    close_price REAL,
    volume INTEGER,
    PRIMARY KEY (trade_date, ticker),
    FOREIGN KEY (ticker) REFERENCES portfolio_mapping(ticker)
);
""")

conn.commit()

# Bulk load raw tables
portfolio_df.to_sql("portfolio_mapping", conn, if_exists="append", index=False)
market_data[
    [
        "trade_date",
        "ticker",
        "open_price",
        "high_price",
        "low_price",
        "close_price",
        "volume",
    ]
].to_sql("historical_prices", conn, if_exists="append", index=False)

# 3. Create SQLite View using LAG() for Native Daily Dollar Changes
cursor.execute("""
CREATE VIEW vw_daily_portfolio_exposure AS
SELECT
    p.trade_date,
    p.ticker,
    m.sector,
    m.shares_held,
    p.close_price,
    (p.close_price * m.shares_held) AS market_value,
    -- Compute previous close price natively via SQL Window Function
    LAG(p.close_price, 1) OVER (PARTITION BY p.ticker ORDER BY p.trade_date) AS prev_close_price,
    -- Compute daily dollar PnL per asset natively via SQL
    (p.close_price - LAG(p.close_price, 1) OVER (PARTITION BY p.ticker ORDER BY p.trade_date)) * m.shares_held AS asset_dollar_pnl
FROM historical_prices p
JOIN portfolio_mapping m ON p.ticker = m.ticker;
""")

conn.commit()

# 4. Pull the SQL-calculated view back into Pandas for statistical modeling loops
exposure_df = pd.read_sql_query(
    "SELECT * FROM vw_daily_portfolio_exposure", conn
)
conn.close()

# Aggregate total portfolio market value and total daily PnL from the SQL view data
portfolio_val = (
    exposure_df.groupby("trade_date")
    .agg({"market_value": "sum", "asset_dollar_pnl": "sum"})
    .reset_index()
    .rename(columns={"asset_dollar_pnl": "daily_pnl"})
    .sort_values("trade_date")
)

portfolio_val["dollar_loss"] = -portfolio_val["daily_pnl"]
portfolio_val = portfolio_val.dropna().reset_index(drop=True)


# 5. Rolling Risk Metrics (VaR 95%, ES 95%, Annualized Dollar Volatility)
window_size = 252
alpha = 0.95

risk_records = []
for i in range(window_size, len(portfolio_val)):
  window_losses = portfolio_val["dollar_loss"].iloc[i - window_size : i].values
  window_pnl = portfolio_val["daily_pnl"].iloc[i - window_size : i].values
  date_val = portfolio_val["trade_date"].iloc[i]

  # Empirical Value-at-Risk (VaR) at alpha = 0.95
  emp_VaR = np.percentile(window_losses, alpha * 100)

  # Empirical Expected Shortfall (ES) at alpha = 0.95
  emp_ES = np.mean(window_losses[window_losses >= emp_VaR])

  # Annualized Volatility
  ann_dollar_vol = np.std(window_pnl) * np.sqrt(252)

  risk_records.append({
      "trade_date": date_val,
      "portfolio_value": portfolio_val["market_value"].iloc[i],
      "daily_pnl": portfolio_val["daily_pnl"].iloc[i],
      "daily_loss": portfolio_val["dollar_loss"].iloc[i],
      "var_95": emp_VaR,
      "es_95": emp_ES,
      "annualized_volatility": ann_dollar_vol,
  })

risk_metrics_df = pd.DataFrame(risk_records)

# 6. Calculate Asset-Level Volatility from SQL View Data
asset_vol_df = (
    exposure_df.groupby("ticker")["asset_dollar_pnl"]
    .agg(annualized_volatility=lambda x: np.std(x.dropna()) * np.sqrt(252))
    .reset_index()
)
asset_vol_df = asset_vol_df.merge(portfolio_df, on="ticker")

# 7. Push Final Analytics Tables back into SQLite
conn = sqlite3.connect(db_name)
risk_metrics_df.to_sql("risk_metrics", conn, if_exists="replace", index=False)
asset_vol_df.to_sql("asset_volatility", conn, if_exists="replace", index=False)
conn.commit()
conn.close()

## PowerBI Python Script

To connect the market risk SQLite database on PowerBI Desktop, use **Python script** to get data and copy and paste the following chunk of code. Make sure the market_risk.db file is downloaded in your computer.

In [ ]:
import sqlite3
import pandas as pd

# Use the absolute path to your database file
db_path = r"C:/Users/YourUsername/Path/To/Your/market_risk.db"  # Update this path!

conn = sqlite3.connect(db_path)

# Load all tables and views into pandas DataFrames for Power BI
historical_prices = pd.read_sql_query("SELECT * FROM historical_prices", conn)
portfolio_mapping = pd.read_sql_query("SELECT * FROM portfolio_mapping", conn)
risk_metrics = pd.read_sql_query("SELECT * FROM risk_metrics", conn)
asset_volatility = pd.read_sql_query(
    "SELECT * FROM asset_volatility", conn
)  # Added for the volatility-by-security chart
exposure_view = pd.read_sql_query(
    "SELECT * FROM vw_daily_portfolio_exposure", conn
)

conn.close()